# 7.2: Convolutions for Images

In [2]:
import torch
from torch import nn
from d2l import torch as d2l

## 7.2.1: The Cross-Correlation Operation

In [3]:
def corr2d(X,K): #@save
    """Compute 2D cross-correlation"""
    h,w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1)) # Build output tensor
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i,j] = (X[i:i + h, j:j + w] * K).sum() # fetch tensors and multiply
    return Y

In [4]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X,K)

tensor([[19., 25.],
        [37., 43.]])

## 7.2.2: Convolutional Layers

In [ ]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

## 7.2.3: Object Edge Detection in Images

In [5]:
X = torch.ones((6,8))
X[:,2:6] = 0 # set middle 4 columns to black (0)
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]])

In [6]:
K = torch.tensor([[1.0, -1.0]]) # height 1, width 2 kernel that computes x_{i,j}-x_{(i+1),j}

In [7]:
Y = corr2d(X,K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]])

In [8]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]])

As we can see, the kernel only detects vertical edges.

## 7.2.4: Learning a Kernel

We will create a convolutional kernel to learn Y, given X.

In [9]:
# Construct a two-dimensional convolutional layer with 1 output channel and a 
# kernel of shape (1,2). For the sake of simplicity, we ignore the bias here
conv2d = nn.LazyConv2d(1, kernel_size=(1,2), bias=False)

# The two-dimensional convolutional layer uses four-dimensional input and
# output in the format of (example, channel, height, width), where the batch
# size (number of examples in the batch) and the number of channels are both 1
X = X.reshape((1,1,6,8))
Y = Y.reshape((1,1,6,7))
lr = 3e-2 # Learning rate

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward() # backprop the MSE
    # Update the kernel
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i+1) % 2 == 0:
        print(f"epoch {i+1}, loss {l.sum():.3f}")

epoch 2, loss 13.075
epoch 4, loss 2.260
epoch 6, loss 0.407
epoch 8, loss 0.079
epoch 10, loss 0.018


c:\Users\Py Torch\.conda\envs\d2l\lib\site-packages\torch\nn\modules\lazy.py:180: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


In [10]:
conv2d.weight.data.reshape((1,2))

tensor([[ 0.9905, -0.9723]])

## 7.2.5: Cross-Correlation and Convolution

To obtain the output of strict convolution operation (i.e. mathematical convolution), just flip the two-dimensional kernel tensor horizontally and vertically, and then perform cross-correlation with the input tensor.

## 7.2.6: Feature Map and Receptive Field


# Summary:

* Core computation required for a convolutional layer is cross-correlation operation.
* This can be computed using a simple nested for-loop
* Convolutional kernels can be learned from data, replacing feature engineering heuristics by evidence-based statistics
* Convolutional layers correspond to receptive fields and feature maps in the brain